# Notebook 08: Complete Training Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChunLI-666/3DGS-from-scratch/blob/develop/notebooks/phase1/08_training_pipeline.ipynb)

---

## Learning Objectives

By the end of this notebook, you will:
1. Understand the complete 3DGS training pipeline
2. Implement a simplified but functional training loop
3. Learn about learning rate scheduling and optimization strategies
4. Understand the role of each component in training
5. Train Gaussians on a simple synthetic scene

**Estimated Time**: 90 minutes

**Prerequisites**: All previous notebooks (00-07)

---

## Setup

In [ ]:
import os
import sys

# Colab setup
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('3DGS-from-scratch'):
        !git clone https://github.com/ChunLI-666/3DGS-from-scratch.git
    os.chdir('3DGS-from-scratch')
    !pip install -q plotly ipywidgets

# Path setup
for path in ['../../src', '../src', './src']:
    full_path = os.path.abspath(path)
    if os.path.exists(os.path.join(full_path, 'gaussian')):
        sys.path.insert(0, full_path)
        break

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from IPython.display import clear_output
import math
from dataclasses import dataclass
from typing import Optional, Dict, List, Tuple
import time

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print("Setup complete!")

## 1. Training Pipeline Overview

### The Complete 3DGS Training Loop

```
┌────────────────────────────────────────────────────────────┐
│                    INITIALIZATION                          │
├────────────────────────────────────────────────────────────┤
│  1. Load point cloud (from SfM: COLMAP)                   │
│  2. Initialize Gaussians at point positions               │
│  3. Set initial scales, opacities, colors/SH              │
│  4. Setup optimizer with per-parameter learning rates     │
└────────────────────────────────────────────────────────────┘
                              │
                              ▼
┌────────────────────────────────────────────────────────────┐
│               TRAINING LOOP (30,000 iterations)            │
├────────────────────────────────────────────────────────────┤
│  For each iteration:                                       │
│    1. Sample random training view                          │
│    2. Render Gaussians from that view                     │
│    3. Compute loss: L1 + λ·D-SSIM                         │
│    4. Backpropagate gradients                             │
│    5. Update parameters (Adam optimizer)                  │
│    6. Densification (every 100 iters, 500-15000)          │
│       - Accumulate gradients                              │
│       - Split/Clone high-gradient Gaussians               │
│       - Prune low-opacity Gaussians                       │
│    7. Opacity reset (every 3000 iters)                    │
│    8. Update learning rate (exponential decay)            │
└────────────────────────────────────────────────────────────┘
                              │
                              ▼
┌────────────────────────────────────────────────────────────┐
│                    OUTPUT                                  │
├────────────────────────────────────────────────────────────┤
│  Optimized Gaussians: positions, scales, rotations,       │
│  opacities, spherical harmonics coefficients              │
└────────────────────────────────────────────────────────────┘
```

In [ ]:
# Configuration dataclass
@dataclass
class TrainingConfig:
    """Configuration for 3DGS training."""
    
    # Training iterations
    max_iterations: int = 30000
    
    # Learning rates
    lr_position: float = 0.00016
    lr_position_final: float = 0.0000016
    lr_scale: float = 0.005
    lr_rotation: float = 0.001
    lr_opacity: float = 0.05
    lr_sh_dc: float = 0.0025
    lr_sh_rest: float = 0.0025 / 20
    
    # Densification
    densify_start: int = 500
    densify_end: int = 15000
    densify_interval: int = 100
    densify_grad_threshold: float = 0.0002
    
    # Pruning
    prune_opacity_threshold: float = 0.005
    
    # Opacity reset
    opacity_reset_interval: int = 3000
    
    # Loss weights
    lambda_dssim: float = 0.2
    
    # Misc
    sh_degree: int = 3
    background: str = 'white'  # 'white' or 'black'
    
    def __post_init__(self):
        self.background_color = torch.ones(3) if self.background == 'white' else torch.zeros(3)


# Print default config
config = TrainingConfig()
print("Default Training Configuration:")
print("=" * 50)
for field in config.__dataclass_fields__:
    value = getattr(config, field)
    if not field.startswith('_'):
        print(f"  {field}: {value}")

## 2. Gaussian Model Class

Let's create a complete Gaussian model that manages all parameters.

In [ ]:
# SH Constants
SH_C0 = 0.28209479177387814

class GaussianModel(nn.Module):
    """
    Complete Gaussian Model for 3DGS training.
    
    Manages all Gaussian parameters with proper activations.
    """
    
    def __init__(self, n_gaussians: int, sh_degree: int = 3):
        super().__init__()
        
        self.n_gaussians = n_gaussians
        self.sh_degree = sh_degree
        self.n_sh_coeffs = (sh_degree + 1) ** 2
        
        # Core parameters (raw, before activation)
        self.means = nn.Parameter(torch.zeros(n_gaussians, 3))
        self.scales_raw = nn.Parameter(torch.zeros(n_gaussians, 3))  # log scale
        self.rotations_raw = nn.Parameter(torch.zeros(n_gaussians, 4))  # quaternion
        self.opacities_raw = nn.Parameter(torch.zeros(n_gaussians))  # logit
        
        # SH coefficients: DC (degree 0) + rest (degrees 1-3)
        self.sh_dc = nn.Parameter(torch.zeros(n_gaussians, 1, 3))  # [N, 1, 3]
        self.sh_rest = nn.Parameter(torch.zeros(n_gaussians, self.n_sh_coeffs - 1, 3))
        
        # Initialize
        self._initialize_parameters()
    
    def _initialize_parameters(self):
        """Initialize parameters to sensible defaults."""
        with torch.no_grad():
            # Identity rotation
            self.rotations_raw.data[:, 0] = 1.0
            
            # Small initial scale
            self.scales_raw.data.fill_(-3.0)  # exp(-3) ≈ 0.05
            
            # Medium opacity
            self.opacities_raw.data.fill_(0.0)  # sigmoid(0) = 0.5
            
            # Gray color
            self.sh_dc.data.fill_(0.0)
    
    def initialize_from_points(
        self,
        points: torch.Tensor,
        colors: Optional[torch.Tensor] = None,
        scale: float = 0.1,
    ):
        """Initialize Gaussians from 3D points."""
        assert points.shape[0] == self.n_gaussians
        
        with torch.no_grad():
            self.means.data = points.clone()
            self.scales_raw.data = torch.log(torch.full((self.n_gaussians, 3), scale))
            
            if colors is not None:
                # Convert RGB to SH DC
                self.sh_dc.data[:, 0] = (colors - 0.5) / SH_C0
    
    @property
    def scales(self) -> torch.Tensor:
        """Activated scales (always positive)."""
        return torch.exp(self.scales_raw)
    
    @property
    def rotations(self) -> torch.Tensor:
        """Normalized quaternions."""
        return F.normalize(self.rotations_raw, dim=-1)
    
    @property
    def opacities(self) -> torch.Tensor:
        """Activated opacities (0-1)."""
        return torch.sigmoid(self.opacities_raw)
    
    @property
    def sh_coefficients(self) -> torch.Tensor:
        """Complete SH coefficients [N, K, 3]."""
        return torch.cat([self.sh_dc, self.sh_rest], dim=1)
    
    def get_covariances(self) -> torch.Tensor:
        """Compute 3D covariance matrices."""
        scales = self.scales
        rotations = self.rotations
        
        # Build rotation matrices from quaternions
        R = self._quaternion_to_rotation_matrix(rotations)
        
        # Build diagonal scaling matrix
        S = torch.diag_embed(scales)  # [N, 3, 3]
        
        # Covariance: Σ = R @ S @ S.T @ R.T
        RS = R @ S
        return RS @ RS.transpose(-1, -2)
    
    def _quaternion_to_rotation_matrix(self, q: torch.Tensor) -> torch.Tensor:
        """Convert quaternions [N, 4] to rotation matrices [N, 3, 3]."""
        q = F.normalize(q, dim=-1)
        w, x, y, z = q[:, 0], q[:, 1], q[:, 2], q[:, 3]
        
        N = q.shape[0]
        R = torch.zeros(N, 3, 3, device=q.device, dtype=q.dtype)
        
        R[:, 0, 0] = 1 - 2*(y*y + z*z)
        R[:, 0, 1] = 2*(x*y - w*z)
        R[:, 0, 2] = 2*(x*z + w*y)
        R[:, 1, 0] = 2*(x*y + w*z)
        R[:, 1, 1] = 1 - 2*(x*x + z*z)
        R[:, 1, 2] = 2*(y*z - w*x)
        R[:, 2, 0] = 2*(x*z - w*y)
        R[:, 2, 1] = 2*(y*z + w*x)
        R[:, 2, 2] = 1 - 2*(x*x + y*y)
        
        return R
    
    def get_parameter_groups(self, config: TrainingConfig) -> List[Dict]:
        """Get parameter groups with different learning rates."""
        return [
            {'params': [self.means], 'lr': config.lr_position, 'name': 'means'},
            {'params': [self.scales_raw], 'lr': config.lr_scale, 'name': 'scales'},
            {'params': [self.rotations_raw], 'lr': config.lr_rotation, 'name': 'rotations'},
            {'params': [self.opacities_raw], 'lr': config.lr_opacity, 'name': 'opacities'},
            {'params': [self.sh_dc], 'lr': config.lr_sh_dc, 'name': 'sh_dc'},
            {'params': [self.sh_rest], 'lr': config.lr_sh_rest, 'name': 'sh_rest'},
        ]
    
    def __len__(self):
        return self.n_gaussians


# Test the model
model = GaussianModel(n_gaussians=100, sh_degree=3)
print(f"GaussianModel created with {len(model)} Gaussians")
print(f"\nParameter shapes:")
for name, param in model.named_parameters():
    print(f"  {name}: {param.shape}")

print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

## 3. Simplified 2D Renderer

For this tutorial, we'll use a simplified 2D renderer to demonstrate the training loop.

In [ ]:
class Simple2DRenderer:
    """
    Simplified 2D Gaussian renderer for educational purposes.
    
    This renderer works directly in 2D to focus on the training loop.
    """
    
    def __init__(self, height: int, width: int, background: torch.Tensor = None):
        self.height = height
        self.width = width
        self.background = background if background is not None else torch.ones(3)
        
        # Create coordinate grid
        y = torch.arange(height, dtype=torch.float32)
        x = torch.arange(width, dtype=torch.float32)
        self.y_grid, self.x_grid = torch.meshgrid(y, x, indexing='ij')
    
    def to(self, device):
        self.background = self.background.to(device)
        self.x_grid = self.x_grid.to(device)
        self.y_grid = self.y_grid.to(device)
        return self
    
    def render(
        self,
        means: torch.Tensor,      # [N, 2]
        scales: torch.Tensor,     # [N, 2]
        opacities: torch.Tensor,  # [N]
        colors: torch.Tensor,     # [N, 3]
        depths: torch.Tensor = None,  # [N] for sorting
    ) -> torch.Tensor:
        """
        Render 2D Gaussians with alpha blending.
        
        Returns:
            Rendered image [H, W, 3]
        """
        N = means.shape[0]
        device = means.device
        
        # Sort by depth (if provided)
        if depths is not None:
            order = torch.argsort(depths)
            means = means[order]
            scales = scales[order]
            opacities = opacities[order]
            colors = colors[order]
        
        # Initialize
        image = torch.zeros(self.height, self.width, 3, device=device)
        transmittance = torch.ones(self.height, self.width, device=device)
        
        for i in range(N):
            mean = means[i]
            scale = scales[i]
            opacity = opacities[i]
            color = colors[i]
            
            # Build 2D covariance (axis-aligned for simplicity)
            cov = torch.diag(scale ** 2 + 1e-6)  # [2, 2]
            cov_inv = torch.linalg.inv(cov)
            
            # Compute Gaussian values
            dx = self.x_grid - mean[0]
            dy = self.y_grid - mean[1]
            
            mahal = (cov_inv[0, 0] * dx * dx + 
                    (cov_inv[0, 1] + cov_inv[1, 0]) * dx * dy +
                    cov_inv[1, 1] * dy * dy)
            
            gaussian_val = torch.exp(-0.5 * mahal)
            alpha = gaussian_val * opacity
            
            # Alpha blending
            weight = transmittance * alpha
            for c in range(3):
                image[:, :, c] = image[:, :, c] + weight * color[c]
            
            transmittance = transmittance * (1 - alpha)
        
        # Add background
        for c in range(3):
            image[:, :, c] = image[:, :, c] + transmittance * self.background[c]
        
        return torch.clamp(image, 0, 1)


# Test renderer
renderer = Simple2DRenderer(64, 64)
means = torch.tensor([[32., 32.], [20., 40.], [45., 25.]])
scales = torch.tensor([[10., 8.], [6., 12.], [8., 8.]])
opacities = torch.tensor([0.8, 0.7, 0.6])
colors = torch.tensor([[1., 0., 0.], [0., 1., 0.], [0., 0., 1.]])

image = renderer.render(means, scales, opacities, colors)

plt.figure(figsize=(5, 5))
plt.imshow(image.numpy())
plt.title('Test Render')
plt.axis('off')
plt.show()

## 4. Loss Functions

In [ ]:
def l1_loss(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    """L1 loss between predicted and target images."""
    return torch.abs(pred - target).mean()


def gaussian_kernel_1d(size: int, sigma: float) -> torch.Tensor:
    """Create 1D Gaussian kernel."""
    x = torch.arange(size, dtype=torch.float32) - size // 2
    kernel = torch.exp(-x**2 / (2 * sigma**2))
    return kernel / kernel.sum()


def ssim(
    pred: torch.Tensor,
    target: torch.Tensor,
    window_size: int = 11,
    sigma: float = 1.5,
) -> torch.Tensor:
    """
    Compute SSIM between predicted and target images.
    
    Args:
        pred: [H, W, C]
        target: [H, W, C]
    
    Returns:
        SSIM value (higher is better)
    """
    C1 = 0.01 ** 2
    C2 = 0.03 ** 2
    
    # Create Gaussian kernel
    kernel_1d = gaussian_kernel_1d(window_size, sigma)
    kernel_2d = kernel_1d.unsqueeze(1) * kernel_1d.unsqueeze(0)
    kernel_2d = kernel_2d.unsqueeze(0).unsqueeze(0)  # [1, 1, H, W]
    kernel_2d = kernel_2d.to(pred.device)
    
    # Reshape for conv2d: [H, W, C] -> [1, C, H, W]
    pred_nhwc = pred.permute(2, 0, 1).unsqueeze(0)
    target_nhwc = target.permute(2, 0, 1).unsqueeze(0)
    
    # Pad
    pad = window_size // 2
    
    # Compute per-channel SSIM
    ssim_vals = []
    for c in range(3):
        p = pred_nhwc[:, c:c+1]
        t = target_nhwc[:, c:c+1]
        
        mu_p = F.conv2d(F.pad(p, (pad, pad, pad, pad), mode='reflect'), kernel_2d)
        mu_t = F.conv2d(F.pad(t, (pad, pad, pad, pad), mode='reflect'), kernel_2d)
        
        mu_p_sq = mu_p ** 2
        mu_t_sq = mu_t ** 2
        mu_pt = mu_p * mu_t
        
        sigma_p_sq = F.conv2d(F.pad(p**2, (pad, pad, pad, pad), mode='reflect'), kernel_2d) - mu_p_sq
        sigma_t_sq = F.conv2d(F.pad(t**2, (pad, pad, pad, pad), mode='reflect'), kernel_2d) - mu_t_sq
        sigma_pt = F.conv2d(F.pad(p*t, (pad, pad, pad, pad), mode='reflect'), kernel_2d) - mu_pt
        
        ssim_map = ((2 * mu_pt + C1) * (2 * sigma_pt + C2)) / \
                   ((mu_p_sq + mu_t_sq + C1) * (sigma_p_sq + sigma_t_sq + C2))
        
        ssim_vals.append(ssim_map.mean())
    
    return torch.stack(ssim_vals).mean()


def combined_loss(
    pred: torch.Tensor,
    target: torch.Tensor,
    lambda_dssim: float = 0.2,
) -> torch.Tensor:
    """
    Combined loss: (1-λ)*L1 + λ*D-SSIM
    
    D-SSIM = (1 - SSIM) / 2
    """
    l1 = l1_loss(pred, target)
    ssim_val = ssim(pred, target)
    dssim = (1 - ssim_val) / 2
    
    return (1 - lambda_dssim) * l1 + lambda_dssim * dssim


# Test loss functions
target = torch.rand(64, 64, 3)
pred_good = target + torch.randn_like(target) * 0.05
pred_bad = target + torch.randn_like(target) * 0.3

print("Loss Function Test:")
print(f"  Good pred: L1={l1_loss(pred_good, target):.4f}, SSIM={ssim(pred_good, target):.4f}")
print(f"  Bad pred:  L1={l1_loss(pred_bad, target):.4f}, SSIM={ssim(pred_bad, target):.4f}")

## 5. Learning Rate Scheduler

In [ ]:
def exponential_lr_decay(
    initial_lr: float,
    final_lr: float,
    current_step: int,
    max_steps: int,
) -> float:
    """
    Compute learning rate with exponential decay.
    
    lr(t) = initial_lr * (final_lr / initial_lr)^(t / max_steps)
    """
    if max_steps == 0:
        return initial_lr
    
    ratio = final_lr / initial_lr
    progress = min(current_step / max_steps, 1.0)
    return initial_lr * (ratio ** progress)


def update_learning_rate(
    optimizer: torch.optim.Optimizer,
    config: TrainingConfig,
    iteration: int,
):
    """
    Update learning rates for optimizer parameter groups.
    
    Only position learning rate decays exponentially.
    """
    for group in optimizer.param_groups:
        if group.get('name') == 'means':
            group['lr'] = exponential_lr_decay(
                config.lr_position,
                config.lr_position_final,
                iteration,
                config.max_iterations,
            )


# Visualize LR schedule
iterations = np.arange(0, 30001, 100)
lrs = [exponential_lr_decay(0.00016, 0.0000016, i, 30000) for i in iterations]

plt.figure(figsize=(10, 4))
plt.semilogy(iterations, lrs)
plt.xlabel('Iteration')
plt.ylabel('Learning Rate (log scale)')
plt.title('Position Learning Rate Schedule')
plt.grid(True, alpha=0.3)
plt.show()

## 6. Simplified 2D Gaussian Model for Training

In [ ]:
class Simple2DGaussianModel(nn.Module):
    """
    Simplified 2D Gaussian model for demonstration.
    """
    
    def __init__(self, n_gaussians: int, height: int, width: int):
        super().__init__()
        
        self.n_gaussians = n_gaussians
        self.height = height
        self.width = width
        
        # Parameters
        self.means = nn.Parameter(torch.zeros(n_gaussians, 2))
        self.scales_raw = nn.Parameter(torch.zeros(n_gaussians, 2))
        self.opacities_raw = nn.Parameter(torch.zeros(n_gaussians))
        self.colors_raw = nn.Parameter(torch.zeros(n_gaussians, 3))
        
        self._initialize()
    
    def _initialize(self):
        with torch.no_grad():
            # Random positions
            self.means.data = torch.rand(self.n_gaussians, 2) * torch.tensor(
                [self.width, self.height], dtype=torch.float32
            )
            
            # Initial scale
            self.scales_raw.data.fill_(math.log(5.0))
            
            # Initial opacity (sigmoid(0) = 0.5)
            self.opacities_raw.data.fill_(0.0)
            
            # Gray color
            self.colors_raw.data.fill_(0.0)
    
    @property
    def scales(self):
        return torch.exp(self.scales_raw)
    
    @property
    def opacities(self):
        return torch.sigmoid(self.opacities_raw)
    
    @property
    def colors(self):
        return torch.sigmoid(self.colors_raw)
    
    def render(self, renderer: Simple2DRenderer) -> torch.Tensor:
        return renderer.render(
            self.means, self.scales, self.opacities, self.colors
        )
    
    def get_parameter_groups(self, lr: float = 0.01):
        return [
            {'params': [self.means], 'lr': lr * 0.5, 'name': 'means'},
            {'params': [self.scales_raw], 'lr': lr * 0.1, 'name': 'scales'},
            {'params': [self.opacities_raw], 'lr': lr * 0.5, 'name': 'opacities'},
            {'params': [self.colors_raw], 'lr': lr, 'name': 'colors'},
        ]


# Test
model_2d = Simple2DGaussianModel(50, 64, 64)
print(f"2D Model with {model_2d.n_gaussians} Gaussians")
print(f"Means range: [{model_2d.means.min():.1f}, {model_2d.means.max():.1f}]")
print(f"Scales range: [{model_2d.scales.min():.2f}, {model_2d.scales.max():.2f}]")

## 7. Create Target Image

In [ ]:
def create_target_image(height: int, width: int) -> torch.Tensor:
    """
    Create a simple target image with geometric shapes.
    """
    image = torch.ones(height, width, 3)  # White background
    
    y_grid, x_grid = torch.meshgrid(
        torch.arange(height, dtype=torch.float32),
        torch.arange(width, dtype=torch.float32),
        indexing='ij'
    )
    
    # Red circle (top-left)
    mask1 = ((x_grid - width * 0.25)**2 + (y_grid - height * 0.3)**2) < (height * 0.15)**2
    image[mask1] = torch.tensor([0.9, 0.2, 0.2])
    
    # Green circle (top-right)
    mask2 = ((x_grid - width * 0.75)**2 + (y_grid - height * 0.3)**2) < (height * 0.12)**2
    image[mask2] = torch.tensor([0.2, 0.8, 0.3])
    
    # Blue ellipse (bottom)
    mask3 = ((x_grid - width * 0.5)**2 / (width * 0.3)**2 + 
             (y_grid - height * 0.7)**2 / (height * 0.12)**2) < 1
    image[mask3] = torch.tensor([0.3, 0.4, 0.9])
    
    # Yellow small circle
    mask4 = ((x_grid - width * 0.5)**2 + (y_grid - height * 0.5)**2) < (height * 0.08)**2
    image[mask4] = torch.tensor([0.95, 0.9, 0.2])
    
    return image


# Create target
HEIGHT, WIDTH = 64, 64
target_image = create_target_image(HEIGHT, WIDTH)

plt.figure(figsize=(5, 5))
plt.imshow(target_image.numpy())
plt.title('Target Image')
plt.axis('off')
plt.show()

## 8. Training Loop

In [ ]:
def train_gaussians(
    target: torch.Tensor,
    n_gaussians: int = 100,
    n_iterations: int = 1000,
    lr: float = 0.05,
    lambda_dssim: float = 0.2,
    visualize_every: int = 100,
):
    """
    Train 2D Gaussians to reconstruct target image.
    """
    height, width = target.shape[:2]
    
    # Initialize
    model = Simple2DGaussianModel(n_gaussians, height, width)
    renderer = Simple2DRenderer(height, width)
    
    # Optimizer
    optimizer = torch.optim.Adam(model.get_parameter_groups(lr))
    
    # Training history
    history = {
        'loss': [],
        'l1': [],
        'ssim': [],
        'images': [],
        'iterations': [],
    }
    
    start_time = time.time()
    
    for i in range(n_iterations):
        optimizer.zero_grad()
        
        # Render
        pred = model.render(renderer)
        
        # Compute loss
        l1 = l1_loss(pred, target)
        ssim_val = ssim(pred, target)
        dssim = (1 - ssim_val) / 2
        loss = (1 - lambda_dssim) * l1 + lambda_dssim * dssim
        
        # Backward
        loss.backward()
        
        # Update
        optimizer.step()
        
        # Record
        history['loss'].append(loss.item())
        history['l1'].append(l1.item())
        history['ssim'].append(ssim_val.item())
        
        if i % visualize_every == 0 or i == n_iterations - 1:
            history['images'].append(pred.detach().clone())
            history['iterations'].append(i)
            print(f"Iter {i:4d}: Loss={loss.item():.4f}, L1={l1.item():.4f}, SSIM={ssim_val.item():.4f}")
    
    elapsed = time.time() - start_time
    print(f"\nTraining complete in {elapsed:.1f}s")
    print(f"Final: Loss={history['loss'][-1]:.4f}, SSIM={history['ssim'][-1]:.4f}")
    
    return model, history


# Train!
model, history = train_gaussians(
    target_image,
    n_gaussians=100,
    n_iterations=500,
    lr=0.1,
    visualize_every=50,
)

In [ ]:
# Visualize training progress
fig = plt.figure(figsize=(16, 8))

# Loss curves
ax1 = fig.add_subplot(2, 4, 1)
ax1.semilogy(history['loss'])
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Loss (log)')
ax1.set_title('Combined Loss')
ax1.grid(True, alpha=0.3)

ax2 = fig.add_subplot(2, 4, 2)
ax2.plot(history['ssim'])
ax2.set_xlabel('Iteration')
ax2.set_ylabel('SSIM')
ax2.set_title('SSIM (higher is better)')
ax2.set_ylim(0, 1)
ax2.grid(True, alpha=0.3)

# Target
ax3 = fig.add_subplot(2, 4, 3)
ax3.imshow(target_image.numpy())
ax3.set_title('Target')
ax3.axis('off')

# Final
ax4 = fig.add_subplot(2, 4, 4)
ax4.imshow(history['images'][-1].numpy())
ax4.set_title(f'Final (SSIM={history["ssim"][-1]:.3f})')
ax4.axis('off')

# Evolution
n_show = min(4, len(history['images']))
indices = [int(i * (len(history['images']) - 1) / (n_show - 1)) for i in range(n_show)]

for plot_idx, img_idx in enumerate(indices):
    ax = fig.add_subplot(2, 4, 5 + plot_idx)
    ax.imshow(history['images'][img_idx].numpy())
    ax.set_title(f'Iter {history["iterations"][img_idx]}')
    ax.axis('off')

plt.tight_layout()
plt.show()

## 9. Visualize Learned Gaussians

In [ ]:
def visualize_gaussians(model, height, width):
    """Visualize learned Gaussian positions, scales, and colors."""
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    means = model.means.detach().numpy()
    scales = model.scales.detach().numpy()
    opacities = model.opacities.detach().numpy()
    colors = model.colors.detach().numpy()
    
    # 1. Gaussian ellipses
    ax = axes[0]
    ax.set_xlim(0, width)
    ax.set_ylim(height, 0)  # Flip y
    
    for i in range(len(means)):
        if opacities[i] > 0.1:  # Only show visible
            ellipse = Ellipse(
                (means[i, 0], means[i, 1]),
                width=scales[i, 0] * 4,
                height=scales[i, 1] * 4,
                fill=True,
                alpha=opacities[i] * 0.5,
                color=colors[i],
            )
            ax.add_patch(ellipse)
    
    ax.set_aspect('equal')
    ax.set_title(f'Learned Gaussians\n(showing {(opacities > 0.1).sum()} / {len(opacities)})')
    
    # 2. Opacity histogram
    ax = axes[1]
    ax.hist(opacities, bins=30, edgecolor='black', alpha=0.7)
    ax.axvline(x=0.1, color='red', linestyle='--', label='Visibility threshold')
    ax.set_xlabel('Opacity')
    ax.set_ylabel('Count')
    ax.set_title('Opacity Distribution')
    ax.legend()
    
    # 3. Scale histogram
    ax = axes[2]
    ax.hist(scales.flatten(), bins=30, edgecolor='black', alpha=0.7)
    ax.set_xlabel('Scale')
    ax.set_ylabel('Count')
    ax.set_title('Scale Distribution')
    
    plt.tight_layout()
    plt.show()


visualize_gaussians(model, HEIGHT, WIDTH)

## 10. Training with More Gaussians

In [ ]:
# Train with more Gaussians for better quality
model_hq, history_hq = train_gaussians(
    target_image,
    n_gaussians=300,
    n_iterations=1000,
    lr=0.08,
    visualize_every=100,
)

In [ ]:
# Compare results
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(target_image.numpy())
axes[0].set_title('Target')
axes[0].axis('off')

axes[1].imshow(history['images'][-1].numpy())
axes[1].set_title(f'100 Gaussians\nSSIM={history["ssim"][-1]:.3f}')
axes[1].axis('off')

axes[2].imshow(history_hq['images'][-1].numpy())
axes[2].set_title(f'300 Gaussians\nSSIM={history_hq["ssim"][-1]:.3f}')
axes[2].axis('off')

# Difference
diff = (history_hq['images'][-1] - target_image).abs().mean(dim=-1)
axes[3].imshow(diff.numpy(), cmap='hot')
axes[3].set_title('Error Map (300 Gaussians)')
axes[3].axis('off')

plt.tight_layout()
plt.show()

visualize_gaussians(model_hq, HEIGHT, WIDTH)

## 11. Summary: Training Pipeline

### Key Components

| Component | Purpose | In This Notebook |
|-----------|---------|------------------|
| **Gaussian Model** | Store and activate parameters | `Simple2DGaussianModel` |
| **Renderer** | Convert Gaussians to images | `Simple2DRenderer` |
| **Loss Function** | Measure reconstruction quality | `combined_loss` (L1 + D-SSIM) |
| **Optimizer** | Update parameters | Adam with per-param LR |
| **LR Scheduler** | Decay learning rate | Exponential decay |
| **Densification** | Adapt Gaussian population | (Covered in Notebook 07) |

### Training Algorithm

```python
# Initialize
gaussians = initialize_from_pointcloud(points)
optimizer = setup_optimizer(gaussians)

for iteration in range(max_iterations):
    # 1. Sample view
    camera = sample_training_camera()
    target = load_ground_truth(camera)
    
    # 2. Render
    pred = render(gaussians, camera)
    
    # 3. Compute loss
    loss = (1 - λ) * L1(pred, target) + λ * DSSIM(pred, target)
    
    # 4. Update
    loss.backward()
    optimizer.step()
    
    # 5. Densification (periodic)
    if should_densify(iteration):
        densify_and_prune(gaussians)
    
    # 6. LR update
    update_learning_rate(optimizer, iteration)
```

### Official 3DGS Hyperparameters

| Parameter | Value |
|-----------|-------|
| Max iterations | 30,000 |
| Position LR | 0.00016 → 0.0000016 |
| Scale LR | 0.005 |
| Rotation LR | 0.001 |
| Opacity LR | 0.05 |
| SH LR | 0.0025 (DC), 0.000125 (rest) |
| λ_DSSIM | 0.2 |
| Densify interval | 100 iters |
| Densify range | 500-15000 |
| Opacity reset | every 3000 |

---

## Key Takeaways

1. 3DGS training is gradient-based optimization
2. Different parameters need different learning rates
3. L1 + D-SSIM loss balances pixel accuracy and perceptual quality
4. More Gaussians generally = better quality (but more memory)
5. Densification adapts Gaussian count to scene complexity

---

## Congratulations!

You've completed Phase 1 of the 3DGS tutorial series! You now understand:

1. ✅ Gaussian distributions and their properties
2. ✅ 3D Gaussian representation (mean, covariance, opacity, color)
3. ✅ Projection and splatting (camera models, Jacobians)
4. ✅ Differentiable rendering
5. ✅ Alpha blending and volume rendering
6. ✅ Spherical Harmonics for view-dependent color
7. ✅ Adaptive density control (split, clone, prune)
8. ✅ Complete training pipeline

### Next Steps

In **Phase 2**, we'll explore:
- Official 3DGS code walkthrough
- Training on real datasets
- SLAM integration (SplaTAM, MonoGS)

Happy Gaussian Splatting!